# 1. Import Libraby x data frame

In [1]:
import pandas as pd
import pyodbc

# 2. Import Data frame from SQL Server

In [2]:
conn = pyodbc.connect( #CONNECTION
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-OI1U6J19\MSSQLSERVER01;"
    "DATABASE=1;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;")
df_associated_rule = pd.read_sql_query('''
                                        
SELECT *
FROM [1].[dbo].[Groceries_dataset]
                                       
    ; ''', conn)
df_associated_rule

<>:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\Bom's PC\AppData\Local\Temp\ipykernel_13292\4199517753.py:3: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  "SERVER=LAPTOP-OI1U6J19\MSSQLSERVER01;"
C:\Users\Bom's PC\AppData\Local\Temp\ipykernel_13292\4199517753.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_associated_rule = pd.read_sql_query('''


,Member_number,Date,itemDescription,Day,Season,Vacation
0,100000000,2015-07-21,tropical fruit,Sun,Winter,True
1,2552,2015-01-05,whole milk,Mon,Summer,False
2,2300,2015-09-19,pip fruit,Tue,Summer,False
3,-1000000,2015-12-12,other vegetables,Wed,Summer,False
4,3037,2015-02-01,whole milk,Thu,Summer,False
...,...,...,...,...,...,...
38779,4471,2014-10-08,sliced cheese,Sat,Winter,True
38780,2022,2014-02-23,candy,Sun,Winter,True
38781,1097,2014-04-16,cake bar,Mon,Winter,True
38782,1510,2014-12-03,fruit/vegetable juice,Tue,Winter,True


# 3. Cleaning data

## 3.1 Remove Duplicate

In [3]:
df_associated_rule = df_associated_rule.drop_duplicates()
df_associated_rule

,Member_number,Date,itemDescription,Day,Season,Vacation
0,100000000,2015-07-21,tropical fruit,Sun,Winter,True
1,2552,2015-01-05,whole milk,Mon,Summer,False
2,2300,2015-09-19,pip fruit,Tue,Summer,False
3,-1000000,2015-12-12,other vegetables,Wed,Summer,False
4,3037,2015-02-01,whole milk,Thu,Summer,False
...,...,...,...,...,...,...
38779,4471,2014-10-08,sliced cheese,Sat,Winter,True
38780,2022,2014-02-23,candy,Sun,Winter,True
38781,1097,2014-04-16,cake bar,Mon,Winter,True
38782,1510,2014-12-03,fruit/vegetable juice,Tue,Winter,True


## 3.2 Detect and remove outliers

In [5]:
# --- Calculate Q1 (25th percentile) and Q3 (75th percentile) ---
Q1 = df_associated_rule['Member_number'].quantile(0.25)
Q3 = df_associated_rule['Member_number'].quantile(0.75)

# --- Calculate IQR ---
IQR = Q3 - Q1
print(f"\nQ1: {Q1}, Q3: {Q3}, IQR: {IQR}")

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_associated_rule = df_associated_rule[(df_associated_rule['Member_number'] >= lower_bound) & (df_associated_rule['Member_number'] <= upper_bound)]


Q1: 2002.0, Q3: 4007.0, IQR: 2005.0


In [7]:
df_associated_rule

,Member_number,Date,itemDescription,Day,Season,Vacation
1,2552,2015-01-05,whole milk,Mon,Summer,False
2,2300,2015-09-19,pip fruit,Tue,Summer,False
4,3037,2015-02-01,whole milk,Thu,Summer,False
5,4941,2015-02-14,rolls/buns,Fri,Summer,False
6,4501,2015-05-08,other vegetables,Sat,Summer,False
...,...,...,...,...,...,...
38779,4471,2014-10-08,sliced cheese,Sat,Winter,True
38780,2022,2014-02-23,candy,Sun,Winter,True
38781,1097,2014-04-16,cake bar,Mon,Winter,True
38782,1510,2014-12-03,fruit/vegetable juice,Tue,Winter,True


## 3.3 Remove Irrelevant Data Using Biz Sense

In [8]:
df_associated_rule = df_associated_rule[['Member_number','Date','itemDescription']]
df_associated_rule

,Member_number,Date,itemDescription
1,2552,2015-01-05,whole milk
2,2300,2015-09-19,pip fruit
4,3037,2015-02-01,whole milk
5,4941,2015-02-14,rolls/buns
6,4501,2015-05-08,other vegetables
...,...,...,...
38779,4471,2014-10-08,sliced cheese
38780,2022,2014-02-23,candy
38781,1097,2014-04-16,cake bar
38782,1510,2014-12-03,fruit/vegetable juice


## 3.4 Standardization Capitalization

## 3.5 Converse data type

In [11]:
df_associated_rule.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38446 entries, 1 to 38783
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Member_number    38446 non-null  int64         
 1   Date             38425 non-null  datetime64[ns]
 2   itemDescription  38425 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 1.2+ MB


In [10]:
df_associated_rule["Date"] = pd.to_datetime(df_associated_rule["Date"])

C:\Users\Bom's PC\AppData\Local\Temp\ipykernel_13292\1626635212.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_associated_rule["Date"] = pd.to_datetime(df_associated_rule["Date"])


## 3.6 Cleaning format

## 3.7 Fix Error

## 3.8 Languagues Translation

## 3.9 Handle missing Value

In [12]:
df_associated_rule.dropna(inplace=True)

C:\Users\Bom's PC\AppData\Local\Temp\ipykernel_13292\980798460.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_associated_rule.dropna(inplace=True)


In [13]:
df_associated_rule

,Member_number,Date,itemDescription
1,2552,2015-01-05,whole milk
2,2300,2015-09-19,pip fruit
4,3037,2015-02-01,whole milk
5,4941,2015-02-14,rolls/buns
6,4501,2015-05-08,other vegetables
...,...,...,...
38779,4471,2014-10-08,sliced cheese
38780,2022,2014-02-23,candy
38781,1097,2014-04-16,cake bar
38782,1510,2014-12-03,fruit/vegetable juice


# 4. Save Clear data into database to Visualize

In [14]:
import pyodbc
conn = pyodbc.connect( #CONNECTION
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-OI1U6J19\MSSQLSERVER01;"
    "DATABASE=1;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;")
cursor = conn.cursor()

# --- Map pandas dtype -> SQL Server type ---
dtype_map = lambda dt: "INT" if pd.api.types.is_integer_dtype(dt) else \
                       "FLOAT" if pd.api.types.is_float_dtype(dt) else \
                       "BIT" if pd.api.types.is_bool_dtype(dt) else \
                       "DATETIME" if pd.api.types.is_datetime64_any_dtype(dt) else \
                       "VARCHAR(255)"

table_name = "Groceries_dataset_clear"

# --- Drop + Create table ---
cursor.execute(f"IF OBJECT_ID('dbo.{table_name}', 'U') IS NOT NULL DROP TABLE dbo.{table_name}")
cols = ", ".join([f"[{c}] {dtype_map(df_associated_rule[c].dtype)}" for c in df_associated_rule.columns])
cursor.execute(f"CREATE TABLE dbo.{table_name} ({cols})")

# --- Insert rows ---
cursor.fast_executemany = True
cursor.executemany(f"INSERT INTO dbo.{table_name} VALUES ({','.join(['?']*len(df_associated_rule.columns))})", df_associated_rule.values.tolist())
conn.commit()

<>:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
<>:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
C:\Users\Bom's PC\AppData\Local\Temp\ipykernel_13292\3649632118.py:4: SyntaxWarning: "\M" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\M"? A raw string is also an option.
  "SERVER=LAPTOP-OI1U6J19\MSSQLSERVER01;"
